# Transformer (DistilBERT) 감정분석 — GLUE/SST-2

In [1]:
!pip uninstall -y torch torchvision torchaudio

# 2. 캐시를 무시하고 CUDA 12.4 버전으로 '강제 재설치' 합니다.
!pip install --no-cache-dir --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3. 나머지 Hugging Face 패키지들도 마저 설치합니다.
!pip -q install -U "datasets>=3.0.1" "transformers>=4.45.2" "accelerate>=1.0.1" "evaluate>=0.4.2"

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 392.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 407.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 331.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 386.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 344.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 315.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

In [2]:
!pip install evaluate

  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (48.9 MB)


런타임 재시작

In [1]:
# 정상적으로 나오는지 확인
import torch, transformers, datasets

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

PyTorch: 2.6.0+cu124
CUDA Available: True
Using device: cuda


In [2]:
from datasets import load_dataset
ds = load_dataset("nyu-mll/glue", "sst2", trust_remote_code=True)
print(ds)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'nyu-mll/glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'nyu-mll/glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this se

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
import torch

# 1. 환경 설정
MODEL = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# 3. 전처리 함수 (return_tensors는 여기서 하지 않고 collator에게 맡깁니다)
def preprocess(ex):
    return tokenizer(ex["sentence"], truncation=True, max_length=256)

# 4. 데이터셋 매핑 (ds가 정의되어 있다고 가정)
# remove_columns에 "label"은 포함하지 않도록 주의하세요!
enc = ds.map(preprocess, batched=True, remove_columns=["sentence", "idx"])

# 5. 데이터 콜레이터 (다이내믹 패딩 적용)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 6. 모델 로드 (이때 발생하는 Warning은 무시해도 됩니다)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
import evaluate
from transformers import TrainingArguments, Trainer
import torch

# 지표 로드
acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def metrics(p):
    predictions, labels = p
    # 가장 높은 확률의 위치
    # .item(): 파이썬 숫자로 변경 (scalar array)
    # predictions의 shape 는 (데이터 개수, 클래스 개수)
    # 예) (데이터 개수, 클래스 개수)  (3,4)
    # axis = -1 마지막 축, 클래스 확률 들 중에서 가장 큰 값이 있는 위치(인덱스) 가져와
    # pred >> (3,)
    preds = predictions.argmax(-1)

    return {
        "acc": acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"]
    }

# TrainingArguments 설정
args = TrainingArguments(
    output_dir="/content/sst2_2025",
    eval_strategy="epoch",          # 수정됨: evaluation_strategy -> eval_strategy
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(), # GPU가 있을 때만 fp16 사용
    report_to="none",
    seed=2025
)
# fp16: 16bit / (default) 32bit float >> 16bit로 낮춤(메모리 절약_2배)
# 참고: A100 이상 돌리고 싶다면? bf16=True 사용 권고 (bf: brain floating point)

# Trainer 초기화
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=enc["train"],
    eval_dataset=enc["validation"],
    data_collator=data_collator,
    compute_metrics=metrics
)

In [ ]:
# TrainArguments()
# 1. 저장 및 출력 경로
# output_dir='./results'
# >> 모델 체크포인트 및 결과 저장 경로

# logging_dir='./logs'
# >> 학습 로그(텐서보드 등) 저장 경로

# 2. 학습 규모 및 하이퍼파라미터
# num_train_epochs=3.0
# >> 전체 데이터셋 반복 학습 횟수 (에폭)

# per_device_train_batch_size=16
# >> GPU 1대당 학습 배치 크기 (OOM 발생 시 축소)

# per_device_eval_batch_size=16
# >> GPU 1대당 검증/평가 배치 크기

# learning_rate=5e-5
# >> 초기 가중치 업데이트 속도 (학습률)

# weight_decay=0.01
# >> 과적합 방지를 위한 L2 정규화 강도

# 3. 평가 및 저장 전략(단위 맞춰주는 것이 좋음)
# eval_strategy='epoch'
# >> 성능 평가 시점 ("no", "steps", "epoch")

# save_strategy'epoch'
# >> 모델 저장 시점 ("no", "steps", "epoch")

# eval_steps=500
# >> eval_strategy="steps"일 때의 평가 주기 (스텝 단위)

# save_steps=500
# >> save_strategy="steps"일 때의 저장 주기 (스텝 단위)

# 4. 베스트 모델 관리
# save_total_limit=2
# >> 디스크 용량 관리를 위해 남겨둘 최대 체크포인트 개수

# load_best_model_at_end=True
# >> 학습 종료 후 최고 성능을 낸 모델을 자동 로드

# metric_for_best_model="loss"
# >> 최고 모델을 판가름할 기준 지표 (loss 또는 accuracy 등)

# greater_is_better=False
# >> 기준 지표가 낮을수록 좋은지(False) 높을수록 좋은지(True)

# 5. 로깅 및 모니터링
# logging_steps=100
# >> 몇 스텝마다 화면에 Loss 로그를 출력할지 설정

# report_to='wandb
# >> 외부 모니터링 툴 연동 ("wandb", "tensorboard", "none")

# 6. 하드웨어 가속 및 효율화
# fp16=True
# >> 16비트 혼합 정밀도 연산 사용 (메모리 절약 + 속도 향상)

# gradient_accumulation_steps=1
# >> 큰 배치를 잘게 쪼개서 가중치를 모아 업데이트 (메모리 부족 시 활용)

In [ ]:
# Trainer() 매개변수
# model
# >> 학습하거나 평가할 Hugging Face 모델 객체(Transformer 모델)

# args
# >> 아래에서 설명할 TrainingArguments 객체(학습 환경 설정)

# train_dataset
# >> 학습에 사용할 Dataset 객체

# eval_dataset
# >> 검증(Validation)에 사용할 Dataset 객체

# tokenizer
# >> 데이터를 배치(Batch) 단위로 패딩할 때 자동으로 텍스트를 처리하기 위해 넣어줌.

# compute_metrics
# >> 모델의 성능을 평가할 함수
# >> 예측값과 실제 정답을 받아 Accuracy, F1-score 등 계산하는 커스텀 함수를 만들어 전달

In [7]:
txt=["This movie was amazing!","Worst film ever."]
inp=tokenizer(txt,return_tensors="pt",padding=True,truncation=True,max_length=256).to(model.device)
with torch.no_grad(): out=torch.softmax(model(**inp).logits,dim=-1).cpu().numpy()
for t,p in zip(txt,out): print(f"{t}\n→ Negative={p[0]:.3f}, Positive={p[1]:.3f}")

This movie was amazing!
→ Negative=0.500, Positive=0.500
Worst film ever.
→ Negative=0.494, Positive=0.506


In [6]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
